# 蒙特卡洛优化算法

## 蒙特卡洛方法概述

蒙特卡洛方法是一类基于随机抽样的计算方法。在优化领域，它通过大量随机采样来探索解空间，寻找最优解。

特点：
- 全局搜索能力
- 实现简单
- 天然并行
- 适用性广

## 偏置引导的蒙特卡洛优化

在NSGABLACK框架中，**偏置系统**是核心特色，它可以将蒙特卡洛的随机搜索引导到更有希望的区域。偏置系统通过以下方式增强蒙特卡洛：

### 1. 偏置引导策略
- **历史偏好偏置**：基于历史最优解区域增加采样密度
- **梯度估计偏置**：通过局部梯度估计指导搜索方向
- **领域知识偏置**：融入问题特定的先验知识

### 2. 自适应偏置强度
- 早期阶段：弱偏置，保持探索多样性
- 中期阶段：逐渐增强偏置，加速收敛
- 后期阶段：强偏置，精细化搜索

### 3. 业务配置示例
- **参数敏感度配置**：根据参数重要性调整偏置
- **业务约束偏置**：自动避开不可行区域
- **成本感知偏置**：考虑评估成本的采样策略

In [ ]:
# 导入必要的库
import numpy as np
import matplotlib.pyplot as plt
import time
from typing import List, Dict, Any, Optional, Tuple
from abc import ABC, abstractmethod
from collections import deque

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

print("环境准备完成")
print(f"NumPy 版本: {np.__version__}")

## 偏置系统基类定义

In [ ]:
class BiasBase(ABC):
    """偏置系统基类"""
    
    def __init__(self, name: str, strength: float = 1.0):
        self.name = name
        self.strength = strength
        self.enabled = True
        self.history = []  # 记录偏置效果
        
    @abstractmethod
    def apply_bias(self, x: np.ndarray, bounds: List[Tuple[float, float]], 
                    context: Dict[str, Any] = None) -> np.ndarray:
        """应用偏置到采样点
        
        Args:
            x: 原始采样点
            bounds: 变量边界
            context: 上下文信息（历史最优、迭代次数等）
            
        Returns:
            偏置后的采样点
        """
        pass
    
    def update_context(self, new_best: Optional[np.ndarray] = None, 
                       improvement: Optional[float] = None):
        """更新偏置上下文"""
        if new_best is not None:
            self.history.append(new_best)
        
    def adapt_strength(self, iteration: int, max_iterations: int):
        """自适应调整偏置强度"""
        # S型曲线：前期弱，中期增强，后期保持
        t = iteration / max_iterations
        self.strength = 1 / (1 + np.exp(-10 * (t - 0.5)))


class HistoricalPreferenceBias(BiasBase):
    """历史偏好偏置 - 向历史最优区域倾斜"""
    
    def __init__(self, strength: float = 1.0, memory_size: int = 10):
        super().__init__("历史偏好偏置", strength)
        self.memory_size = memory_size
        self.elite_archive = deque(maxlen=memory_size)
        
    def apply_bias(self, x: np.ndarray, bounds: List[Tuple[float, float]], 
                    context: Dict[str, Any] = None) -> np.ndarray:
        if not self.elite_archive or self.strength < 0.1:
            return x
        
        # 计算向精英中心的偏置
        elite_center = np.mean(list(self.elite_archive), axis=0)
        bias_direction = elite_center - x
        
        # 添加随机扰动避免过早收敛
        noise = np.random.randn(len(x)) * 0.1
        
        # 应用偏置
        biased_x = x + self.strength * (bias_direction + noise)
        
        # 确保在边界内
        for i, (low, high) in enumerate(bounds):
            biased_x[i] = np.clip(biased_x[i], low, high)
        
        return biased_x
    
    def update_context(self, new_best: Optional[np.ndarray] = None, 
                       improvement: Optional[float] = None):
        if new_best is not None:
            self.elite_archive.append(new_best.copy())


class GradientEstimationBias(BiasBase):
    """梯度估计偏置 - 基于局部梯度估计引导搜索"""
    
    def __init__(self, strength: float = 1.0, epsilon: float = 1e-5):
        super().__init__("梯度估计偏置", strength)
        self.epsilon = epsilon
        self.last_evaluations = []  # 存储最近的评估结果
        
    def apply_bias(self, x: np.ndarray, bounds: List[Tuple[float, float]], 
                    context: Dict[str, Any] = None) -> np.ndarray:
        if len(self.last_evaluations) < 2:
            return x
        
        # 使用最近的评估估计梯度方向
        gradient = self._estimate_gradient(x)
        
        # 沿负梯度方向移动（最小化）
        biased_x = x - self.strength * gradient
        
        # 确保在边界内
        for i, (low, high) in enumerate(bounds):
            biased_x[i] = np.clip(biased_x[i], low, high)
        
        return biased_x
    
    def _estimate_gradient(self, x: np.ndarray) -> np.ndarray:
        """估计梯度方向"""
        if len(self.last_evaluations) < 2:
            return np.zeros_like(x)
        
        # 简单的梯度估计：基于最近两个点的差异
        last_x, last_f = self.last_evaluations[-1]
        prev_x, prev_f = self.last_evaluations[-2]
        
        # 避免除零
        diff = last_x - prev_x
        grad = np.zeros_like(diff)
        for i in range(len(diff)):
            if abs(diff[i]) > self.epsilon:
                grad[i] = (last_f - prev_f) / diff[i]
        
        return grad
    
    def record_evaluation(self, x: np.ndarray, f: float):
        """记录评估结果"""
        self.last_evaluations.append((x.copy(), f))
        # 保持最近的评估
        if len(self.last_evaluations) > 10:
            self.last_evaluations.pop(0)


class BusinessConstraintBias(BiasBase):
    """业务约束偏置 - 处理实际业务中的约束和偏好"""
    
    def __init__(self, strength: float = 1.0, constraints: Dict[str, Any] = None):
        super().__init__("业务约束偏置", strength)
        self.constraints = constraints or {}
        
    def apply_bias(self, x: np.ndarray, bounds: List[Tuple[float, float]], 
                    context: Dict[str, Any] = None) -> np.ndarray:
        biased_x = x.copy()
        
        # 处理预定义的业务约束
        if 'budget_limit' in self.constraints:
            # 预算约束：某些参数组合超出预算时调整
            budget_factor = self.constraints['budget_limit']
            if self._estimate_cost(x) > budget_factor:
                # 按比例缩放
                biased_x = biased_x * (budget_factor / self._estimate_cost(x))
        
        if 'priority_params' in self.constraints:
            # 参数优先级：某些参数更重要
            priority = self.constraints['priority_params']
            for i, p in enumerate(priority):
                if p == 'high':
                    # 高优先级参数，减少随机性
                    biased_x[i] = x[i] + self.strength * 0.1 * np.random.randn()
                elif p == 'low':
                    # 低优先级参数，可以更大胆探索
                    biased_x[i] = x[i] + self.strength * 0.5 * np.random.randn()
        
        if 'forbidden_regions' in self.constraints:
            # 禁区：避开某些参数区域
            forbidden = self.constraints['forbidden_regions']
            for region in forbidden:
                if self._in_region(biased_x, region):
                    # 将点推出禁区
                    biased_x = self._push_out_region(biased_x, region, bounds)
        
        # 确保在边界内
        for i, (low, high) in enumerate(bounds):
            biased_x[i] = np.clip(biased_x[i], low, high)
        
        return biased_x
    
    def _estimate_cost(self, x: np.ndarray) -> float:
        """估算配置成本"""
        # 简化的成本模型：参数平方和
        return np.sum(x**2)
    
    def _in_region(self, x: np.ndarray, region: Dict[str, Any]) -> bool:
        """检查点是否在禁区内"""
        if 'center' in region and 'radius' in region:
            return np.linalg.norm(x - region['center']) < region['radius']
        return False
    
    def _push_out_region(self, x: np.ndarray, region: Dict[str, Any], 
                         bounds: List[Tuple[float, float]]) -> np.ndarray:
        """将点推出禁区"""
        if 'center' in region and 'radius' in region:
            direction = x - region['center']
            if np.linalg.norm(direction) > 0:
                direction = direction / np.linalg.norm(direction)
                return region['center'] + direction * (region['radius'] + 0.1)
        return x

## 偏置引导的蒙特卡洛优化器

In [ ]:
class BiasedMonteCarloOptimizer:
    """偏置引导的蒙特卡洛优化器"""
    
    def __init__(self, max_evals=1000, biases: List[BiasBase] = None, seed=None):
        self.max_evals = max_evals
        self.seed = seed
        self.biases = biases or []
        
        # 优化状态
        self.best_x = None
        self.best_f = float('inf')
        self.history = []
        self.evaluation_count = 0
        
        # 性能统计
        self.bias_contributions = {bias.name: 0 for bias in self.biases}
        
        if seed is not None:
            np.random.seed(seed)
            
        print(f"创建偏置引导蒙特卡洛优化器")
        print(f"  最大评估次数: {max_evals}")
        print(f"  激活偏置: {[bias.name for bias in self.biases]}")
        if seed:
            print(f"  随机种子: {seed}")
    
    def generate_biased_sample(self, n_var: int, bounds: List[Tuple[float, float]],
                               iteration: int) -> np.ndarray:
        """生成偏置引导的采样点"""
        # 生成基础随机采样点
        x = np.array([np.random.uniform(low, high) for low, high in bounds])
        
        # 构建上下文信息
        context = {
            'iteration': iteration,
            'best_x': self.best_x,
            'best_f': self.best_f,
            'evaluation_count': self.evaluation_count
        }
        
        # 依次应用各个偏置
        for bias in self.biases:
            if bias.enabled:
                old_x = x.copy()
                x = bias.apply_bias(x, bounds, context)
                
                # 记录偏置贡献
                if np.linalg.norm(x - old_x) > 1e-6:
                    self.bias_contributions[bias.name] += 1
        
        return x
    
    def optimize(self, func, n_var, bounds):
        """优化函数
        
        Args:
            func: 目标函数
            n_var: 变量维度
            bounds: 每个变量的边界 [(low1, high1), (low2, high2), ...]
        """
        print(f"\n开始偏置引导优化 {n_var}维函数...")
        start_time = time.time()
        
        for i in range(self.max_evals):
            # 自适应调整偏置强度
            for bias in self.biases:
                bias.adapt_strength(i, self.max_evals)
            
            # 生成偏置引导的采样点
            x = self.generate_biased_sample(n_var, bounds, i)
            
            # 评估
            f = func(x)
            self.evaluation_count += 1
            
            # 记录评估（用于梯度估计偏置）
            for bias in self.biases:
                if isinstance(bias, GradientEstimationBias):
                    bias.record_evaluation(x, f)
            
            # 更新最优解
            if f < self.best_f:
                improvement = self.best_f - f
                self.best_f = f
                self.best_x = x.copy()
                
                # 更新偏置上下文
                for bias in self.biases:
                    bias.update_context(self.best_x, improvement)
            
            # 记录历史
            self.history.append(self.best_f)
            
            # 输出进度
            if (i + 1) % 200 == 0:
                print(f"  评估: {i+1}/{self.max_evals}, "
                      f"最优值: {self.best_f:.4f}, "
                      f"活跃偏置: {sum(1 for b in self.biases if b.enabled and b.strength > 0.1)}")
        
        end_time = time.time()
        print(f"\n优化完成！")
        print(f"  耗时: {end_time-start_time:.2f}秒")
        print(f"  最优值: {self.best_f:.4f}")
        print(f"  最优解: {self.best_x}")
        
        # 输出偏置统计
        print("\n偏置贡献统计：")
        for bias_name, count in self.bias_contributions.items():
            percentage = count / self.max_evals * 100
            print(f"  {bias_name}: {count}次 ({percentage:.1f}%)")
        
        return self.best_x, self.best_f


# 定义测试函数
def sphere(x):
    """Sphere函数 - 最简单的测试函数"""
    return sum(xi**2 for xi in x)

def rastrigin(x):
    """Rastrigin函数 - 多峰函数"""
    A = 10
    n = len(x)
    return A * n + sum(xi**2 - A * np.cos(2 * np.pi * xi) for xi in x)


# 创建业务配置示例
print("业务配置示例：")
print("="*50)

# 示例1：云服务器配置优化
cloud_config = {
    'budget_limit': 1000,  # 预算限制
    'priority_params': ['high', 'medium', 'low', 'low'],  # 参数优先级
    'forbidden_regions': [
        {'center': np.array([8, 8, 8, 8]), 'radius': 2}  # 不稳定配置区域
    ]
}

print("\n云服务器配置优化场景：")
print("- 参数1: CPU核心数 (高优先级)")
print("- 参数2: 内存大小 (中优先级)")
print("- 参数3: 存储空间 (低优先级)")
print("- 参数4: 带宽 (低优先级)")
print(f"- 预算限制: {cloud_config['budget_limit']}")
print("- 禁区: 高负载不稳定配置")

## 对比实验：标准蒙特卡洛 vs 偏置引导蒙特卡洛

In [ ]:
# 实验设置
n_var = 5
bounds = [(-5.12, 5.12)] * n_var
max_evals = 3000
seed = 42

print("\n对比实验：标准蒙特卡洛 vs 偏置引导蒙特卡洛")
print("="*60)

# 1. 标准蒙特卡洛（无偏置）
print("\n1. 标准蒙特卡洛优化：")
print("-"*30)
standard_mc = BiasedMonteCarloOptimizer(max_evals=max_evals, biases=[], seed=seed)
standard_best_x, standard_best_f = standard_mc.optimize(rastrigin, n_var, bounds)

# 2. 偏置引导蒙特卡洛
print("\n\n2. 偏置引导蒙特卡洛优化：")
print("-"*30)

# 创建偏置组合
biases = [
    HistoricalPreferenceBias(strength=0.5, memory_size=20),
    GradientEstimationBias(strength=0.3),
    BusinessConstraintBias(strength=0.2, constraints=cloud_config)
]

biased_mc = BiasedMonteCarloOptimizer(max_evals=max_evals, biases=biases, seed=seed)
biased_best_x, biased_best_f = biased_mc.optimize(rastrigin, n_var, bounds)

# 3. 不同偏置组合对比
print("\n\n3. 单一偏置效果对比：")
print("-"*30)

single_bias_results = {}
bias_configs = [
    ("仅历史偏置", [HistoricalPreferenceBias(strength=0.7)]),
    ("仅梯度偏置", [GradientEstimationBias(strength=0.5)]),
    ("仅业务偏置", [BusinessConstraintBias(strength=0.5, constraints=cloud_config)])
]

for config_name, bias_list in bias_configs:
    print(f"\n{config_name}：")
    optimizer = BiasedMonteCarloOptimizer(max_evals=max_evals, biases=bias_list, seed=seed)
    best_x, best_f = optimizer.optimize(rastrigin, n_var, bounds)
    single_bias_results[config_name] = best_f

# 可视化对比
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. 收敛曲线对比
ax1 = axes[0, 0]
ax1.plot(standard_mc.history, 'b-', label='标准蒙特卡洛', linewidth=2)
ax1.plot(biased_mc.history, 'r-', label='偏置引导蒙特卡洛', linewidth=2)
ax1.set_xlabel('评估次数')
ax1.set_ylabel('最优目标值')
ax1.set_title('收敛曲线对比')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_yscale('log')

# 2. 最终结果对比
ax2 = axes[0, 1]
methods = ['标准蒙特卡洛', '偏置引导蒙特卡洛'] + list(single_bias_results.keys())
results = [standard_best_f, biased_best_f] + list(single_bias_results.values())
colors = ['blue', 'red', 'green', 'orange', 'purple']

bars = ax2.bar(methods, results, alpha=0.7, color=colors)
ax2.set_ylabel('最终目标值')
ax2.set_title('优化结果对比')
ax2.grid(True, alpha=0.3, axis='y')
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=15, ha='right')

# 添加数值标签
for bar, val in zip(bars, results):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height(), 
             f'{val:.2f}', ha='center', va='bottom')

# 3. 偏置贡献度
ax3 = axes[1, 0]
if biased_mc.bias_contributions:
    bias_names = list(biased_mc.bias_contributions.keys())
    contributions = list(biased_mc.bias_contributions.values())
    
    wedges, texts, autotexts = ax3.pie(contributions, labels=bias_names, 
                                         autopct='%1.1f%%', startangle=90)
    ax3.set_title('偏置贡献度分布')
    
    # 美化文字
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontweight('bold')

# 4. 偏置强度演化
ax4 = axes[1, 1]
# 模拟偏置强度演化
iterations = range(max_evals)
for bias in biases:
    strengths = []
    for i in iterations:
        t = i / max_evals
        strength = 1 / (1 + np.exp(-10 * (t - 0.5)))
        strengths.append(strength)
    ax4.plot(iterations, strengths, label=bias.name, linewidth=2)

ax4.set_xlabel('评估次数')
ax4.set_ylabel('偏置强度')
ax4.set_title('偏置强度自适应演化')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 性能总结
print("\n\n性能总结：")
print("="*50)
improvement = (standard_best_f - biased_best_f) / standard_best_f * 100
print(f"偏置引导改进: {improvement:.2f}%")
print(f"\n最优配置: {biased_best_x}")

# 业务分析
print("\n业务配置分析：")
print(f"CPU核心数: {biased_best_x[0]:.2f}")
print(f"内存大小: {biased_best_x[1]:.2f}")
print(f"存储空间: {biased_best_x[2]:.2f}")
print(f"带宽: {biased_best_x[3]:.2f}")
estimated_cost = np.sum(biased_best_x**2)
print(f"估算成本: {estimated_cost:.2f} (预算限制: {cloud_config['budget_limit']})")

## 偏置在蒙特卡洛中的核心作用

### 1. 从随机到智能的转变
- **纯随机**：可能永远找不到最优解
- **弱偏置**：引导但保持探索
- **强偏置**：快速收敛但可能陷入局部
- **自适应偏置**：平衡探索与利用

### 2. 偏置类型的互补性
- **历史偏置**：记住好位置
- **梯度偏置**：知道方向
- **业务偏置**：遵守规则
- **组合使用**：发挥各自优势

### 3. 实际业务价值
- **减少评估次数**：节省计算资源
- **满足业务约束**：确保方案可行
- **快速收敛**：提高决策效率
- **可解释性**：理解优化过程

## 使用建议

1. **简单问题**：使用历史偏置即可
2. **复杂问题**：组合多种偏置
3. **约束问题**：必须包含业务偏置
4. **动态环境**：使用自适应偏置强度